*0.2 Math / ML basics*

# t-SNE

**The situation.** A reviewer asks for "the t-SNE plot" — it is the picture in every embedding paper and many teams expect it. It does the same job as UMAP (islands from neighbourhoods) with an older method: slower, no way to place *new* points onto an existing picture, but very good at pulling tight clusters apart.

**t-SNE.** For each pair of points it compares "how likely are these neighbours in the full space" with "how likely in the 2-D picture", and moves the 2-D points until the two agree. `perplexity` is roughly how many neighbours each point considers.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import numpy as np
from openai import OpenAI
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score

templates = {
    "billing": ["charged twice for {x}", "refund for {x} not received", "invoice for {x} is wrong"],
    "technical": ["{x} page will not load", "error 500 when opening {x}", "app crashes on {x}"],
    "account": ["reset password for {x}", "change email on {x}", "delete my {x} account"],
}
texts = []
labels = []
for label, patterns in templates.items():
    for pattern in patterns:
        for x in (
            "order",
            "the dashboard",
            "my plan",
            "the mobile app",
            "the report",
            "the team workspace",
            "billing",
            "settings",
            "the export",
            "the API",
        ):
            texts.append(pattern.format(x=x))
            labels.append(label)

client = OpenAI(timeout=60)
vectors = []
for item in client.embeddings.create(model="text-embedding-3-small", input=texts).data:
    vectors.append(item.embedding)
vectors = np.array(vectors, dtype=np.float32)

for perplexity in (5, 30):
    points = TSNE(n_components=2, perplexity=perplexity, random_state=0, init="pca").fit_transform(
        vectors
    )
    found = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(points)
    print(
        
            f"perplexity {perplexity:>2}: islands vs true topics agreement = "
            f"{adjusted_rand_score(labels, found):.2f}"
        
    )
assert adjusted_rand_score(labels, found) > 0.7

perplexity  5: islands vs true topics agreement = 1.00
perplexity 30: islands vs true topics agreement = 0.72


**Reading the output.** Both perplexities recover the three topics. The picture changes with perplexity — that is normal and why one should try more than one value.

**The rule to remember.** t-SNE and UMAP answer the same question ("what groups are there?"). Prefer UMAP for speed and for placing new points; use t-SNE when a reviewer or a tight-cluster picture calls for it.

| Use it when | Don't when | Instead use |
|---|---|---|
| a one-off picture of ≤ 10k embeddings | new points must be placed on an existing picture; datasets above ~50k | UMAP |

**Watch out**
- No `transform` for new data — every new picture is a fresh fit, and it will look different.
- Distances between islands are meaningless; so is island size. Only "these are together" is real.
- `init="pca"` and a fixed `random_state` make results repeatable enough to compare settings.